# BrightPath Case Study 3: Monthly Financial Performance Review
**Machine Test: Data Analyst | Pandas Series**

This notebook answers tasks A1–F2 from the BrightPath Software case study. Financial values are in ₹ lakh.

## Setup and dataset

In [ ]:
import pandas as pd

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
revenue_values = [42, 45, 50, 48, 55, 60, 58, 62, 65, 70, 75, 85]
expense_values = [38, 40, 42, 47, 50, 52, 55, 60, 61, 64, 66, 72]

## Part A — Series construction and vectorised thinking

### A1. Create Revenue and Expenses Series

In [ ]:
revenue = pd.Series(revenue_values, index=months, name="Revenue")
expenses = pd.Series(expense_values, index=months, name="Expenses")

display(revenue)
display(expenses)

### A2. Python list addition

In [ ]:
a = [42, 45, 50]
b = [38, 40, 42]
print(a + b)

**Explanation:** Python list `+` concatenates the lists, producing `[42, 45, 50, 38, 40, 42]`. It does not subtract expenses from revenue or calculate month-by-month profit.

### A3. Profit using a loop

In [ ]:
profit_loop_values = []
for i in range(len(revenue_values)):
    profit_loop_values.append(revenue_values[i] - expense_values[i])

profit_loop = pd.Series(profit_loop_values, index=months, name="Profit (loop)")
display(profit_loop)

### A4. Profit using one Series expression

In [ ]:
profit = (revenue - expenses).rename("Profit")
display(profit)
print("Loop and Series results match:", profit.equals(profit_loop.rename("Profit")))

## Part B — Profit and margin

### B1–B2. Monthly profit and margin

In [ ]:
profit = (revenue - expenses).rename("Profit")
margin = ((profit / revenue) * 100).round(1).rename("Margin (%)")

monthly_summary = pd.concat([revenue, expenses, profit, margin], axis=1)
display(monthly_summary)

### B3. Annual totals and overall margin

In [ ]:
total_revenue = revenue.sum()
total_expenses = expenses.sum()
total_profit = profit.sum()
overall_margin = round(total_profit / total_revenue * 100, 1)

print(f"Total revenue: ₹{total_revenue} lakh")
print(f"Total expenses: ₹{total_expenses} lakh")
print(f"Total profit: ₹{total_profit} lakh")
print(f"Overall margin: {overall_margin}%")

## Part C — Quarterly and half-yearly analysis

### C1. Quarterly profit using `loc`

In [ ]:
quarters = {
    "Q1 (Jan–Mar)": profit.loc["Jan":"Mar"].sum(),
    "Q2 (Apr–Jun)": profit.loc["Apr":"Jun"].sum(),
    "Q3 (Jul–Sep)": profit.loc["Jul":"Sep"].sum(),
    "Q4 (Oct–Dec)": profit.loc["Oct":"Dec"].sum(),
}
display(pd.Series(quarters, name="Quarterly profit (₹ lakh)"))
print("Strongest quarter:", max(quarters, key=quarters.get))

### C2. Half-year profit using `iloc`

In [ ]:
h1_profit = profit.iloc[0:6].sum()
h2_profit = profit.iloc[6:12].sum()
print(f"H1 (Jan–Jun) profit: ₹{h1_profit} lakh")
print(f"H2 (Jul–Dec) profit: ₹{h2_profit} lakh")

### C3. Festive months (Oct, Nov, Dec)

In [ ]:
display(profit.loc[["Oct", "Nov", "Dec"]])

### C4. `iloc` equivalent of `loc["Apr":"Jun"]`

The equivalent positional slice is `profit.iloc[3:6]`. `loc` includes the ending label `"Jun"`, while `iloc` excludes the ending position, so the stop position is `6`.

In [ ]:
display(profit.iloc[3:6])

## Part D — Filter and rank

### D1. Months with margin above 10%

In [ ]:
display(margin[margin > 10])

### D2. Months with profit below ₹4 lakh

In [ ]:
display(profit[profit < 4])

### D3. Best and worst months using boolean conditions

In [ ]:
best_profit = profit.max()
worst_profit = profit.min()
print("Best month(s):")
display(profit[profit == best_profit])
print("Worst month(s):")
display(profit[profit == worst_profit])

### D4. Top 3 months by profit

In [ ]:
display(profit.sort_values(ascending=False).head(3))

### D5. Average monthly profit and months above average

In [ ]:
average_profit = profit.mean()
print(f"Average monthly profit: ₹{average_profit:.2f} lakh")
display(profit[profit > average_profit])

## Part E — Business rules with `apply()`

### E1. Assign profit status

In [ ]:
def profit_status(p):
    if p >= 8:
        return "Strong"
    elif p >= 4:
        return "Stable"
    else:
        return "Weak"

status = profit.apply(profit_status).rename("Status")
display(status)

### E2. Count and percentage in each status

In [ ]:
status_counts = status.value_counts().reindex(["Strong", "Stable", "Weak"], fill_value=0)
status_percentages = (status_counts / len(status) * 100).round(1)
status_summary = pd.DataFrame({
    "Months": status_counts,
    "Percentage (%)": status_percentages
})
display(status_summary)

### E3. Calculate tax payable at 25% of positive profit

In [ ]:
def tax_payable(p):
    return p * 0.25 if p > 0 else 0

tax = profit.apply(tax_payable).rename("Tax (₹ lakh)")
display(tax)

### E4. Total tax and post-tax profit

In [ ]:
total_tax = tax.sum()
post_tax_profit = total_profit - total_tax
print(f"Total estimated tax: ₹{total_tax:.2f} lakh")
print(f"Annual post-tax profit: ₹{post_tax_profit:.2f} lakh")

## Part F — Index alignment and missing data

### F1. Remove December revenue and subtract expenses

In [ ]:
rev_partial = revenue.iloc[0:11]
partial_difference = (rev_partial - expenses).rename("Partial revenue - expenses")
display(partial_difference)

**Observations:**
1. January through November have calculated differences because their month labels align.
2. December is `NaN` because `rev_partial` has no December value, while `expenses` still has December. Pandas aligns Series by index labels.

### F2. Why `NaN` appears and how to handle December

Pandas returns `NaN` because December revenue is missing from the partial Series; it does not assume a value. For a CFO report, retrieve and validate the actual December revenue before finalising the report. If it cannot be obtained, clearly mark December profit and any totals that depend on it as incomplete—do not silently replace missing revenue with zero or an estimate.

## Final CFO summary

In [ ]:
print("BRIGHTPATH SOFTWARE — FINANCIAL SUMMARY")
print(f"Annual revenue: ₹{total_revenue} lakh")
print(f"Annual expenses: ₹{total_expenses} lakh")
print(f"Annual profit before tax: ₹{total_profit} lakh")
print(f"Overall profit margin: {overall_margin}%")
print(f"Estimated tax (25% of positive monthly profits): ₹{total_tax:.2f} lakh")
print(f"Post-tax profit: ₹{post_tax_profit:.2f} lakh")
print(f"Strongest quarter: {max(quarters, key=quarters.get)}")
print("Note: These results use the case-study dataset provided.")